# Hadoop MapReduce en Google Cloud Dataproc

**Proyecto de Bases de Datos II — Big Data**

Ejecucion de tres programas incluidos en `hadoop-mapreduce-examples.jar` sobre un clu&#769;ster administrado de Google Cloud Dataproc, usando **HDFS** como sistema de archivos distribuido para la entrada y la salida.

| Programa | Dataset | Objetivo |
|---|---|---|
| `wordmean` | Don Quijote de la Mancha (Project Gutenberg) | Longitud media de palabra |
| `secondarysort` | MovieLens 100K (GroupLens) | Ordenamiento secundario |
| `terasort` | Generado por `teragen` | Benchmark de ordenamiento distribuido |

> **Como ejecutar este notebook:** abrirlo en JupyterLab del nodo maestro del clu&#769;ster
> (Consola de Google Cloud &rarr; Dataproc &rarr; cluster &rarr; Interfaces web &rarr; JupyterLab)
> y usar *Run All*. Las celdas usan `!` para lanzar comandos de shell en el nodo maestro.
> El notebook se guarda automa&#769;ticamente en el bucket de staging del clu&#769;ster.

---
## 0. Entorno de ejecucion

| Componente | Configuracion |
|---|---|
| Cluster | `cluster-bigdata-v2` |
| Region | `us-east4` |
| Imagen | `2.3-debian12` |
| Maestro | 1 x `n2-highmem-4` (50 GB) |
| Workers | 2 x `n2-standard-4` (50 GB) |
| Bucket | `bigdata-2026-02` |

### 0.1 Version de Hadoop

In [ ]:
!hadoop version | head -3

Hadoop 3.3.6
Source code repository https://bigdataoss-internal.googlesource.com/third_party/apache/hadoop -r 8424d9552b6a186a1e0adc439510b10505b5714b
Compiled by bigtop on 2026-08-30T07:34Z


### 0.2 El JAR de ejemplos

El archivo ya viene preinstalado en cada nodo del cluster; no hace falta descargarlo de Maven Central.
En realidad es un **enlace simbolico**, y esa indireccion revela la version exacta de Hadoop.

In [ ]:
!ls -l /usr/lib/hadoop-mapreduce/hadoop-mapreduce-examples.jar

lrwxrwxrwx 1 root root 35 Aug 30 07:20 /usr/lib/hadoop-mapreduce/hadoop-mapreduce-examples.jar -> hadoop-mapreduce-examples-3.3.6.jar


### 0.3 Programas disponibles en el JAR

Ejecutar el JAR sin argumentos lista todos los ejemplos. De aqui se seleccionaron los tres de
esta practica, excluyendo `wordcount` y `grep` segun el enunciado.

In [ ]:
!hadoop jar /usr/lib/hadoop-mapreduce/hadoop-mapreduce-examples.jar

An example program must be given as the first argument.
Valid program names are:
  aggregatewordcount: An Aggregate based map/reduce program that counts the words in the input files.
  aggregatewordhist: An Aggregate based map/reduce program that computes the histogram of the words in the input files.
  bbp: A map/reduce program that uses Bailey-Borwein-Plouffe to compute exact digits of Pi.
  dbcount: An example job that count the pageview counts from a database.
  distbbp: A map/reduce program that uses a BBP-type formula to compute exact bits of Pi.
  grep: A map/reduce program that counts the matches of a regex in the input.
  join: A job that effects a join over sorted, equally partitioned datasets
  multifilewc: A job that counts words from several files.
  pentomino: A map/reduce tile laying program to find solutions to pentomino problems.
  pi: A map/reduce program that estimates Pi using a quasi-Monte Carlo method.
  randomtextwriter: A map/reduce program that writes 10GB of r

### 0.4 Estado de HDFS

Nota sobre rutas: la terminal de JupyterLab corre como `root`, por lo que se usan rutas fijas
(`/tarea/...`) en lugar de `/user/$(whoami)/...` para evitar inconsistencias entre sesiones.

In [ ]:
!hdfs dfsadmin -report | head -12

Configured Capacity: 168585527296 (157.01 GB)
Present Capacity: 126396292905 (117.72 GB)
DFS Remaining: 126067884032 (117.41 GB)
DFS Used: 328408873 (313.20 MB)
DFS Used%: 0.26%
Replicated Blocks:
	Under replicated blocks: 1
	Blocks with corrupt replicas: 0
	Missing blocks: 0
	Missing blocks (with replication factor 1): 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0


---
# 1. Programa 1 &mdash; `wordmean`

## 1.1 Objetivo

Calcular la **longitud promedio de las palabras** de un conjunto de archivos de texto.

- **Map:** por cada palabra emite dos pares clave-valor: `count` &rarr; 1, y `length` &rarr; numero de caracteres.
- **Reduce:** suma ambos acumuladores.
- Finalmente el programa lee su propio archivo de salida y divide `length / count`.

## 1.2 Por que este dataset

La primera opcion fueron los archivos `LICENSE.txt` y `NOTICE.txt` que vienen con Hadoop, pero se
descartaron: son texto legal, pequenos y poco representativos como conjunto de datos.

Se eligio **"Don Quijote de la Mancha"** (Project Gutenberg, eBook #2000) porque:

1. Es de **dominio publico**, descargable y citable sin problemas de licencia.
2. Esta en **texto plano UTF-8**, que es justo lo que espera `TextInputFormat`; las tildes y la "n&#771;" se procesan sin configuracion extra.
3. Es texto natural real con gran variedad lexica, asi que la media tiene una interpretacion linguistica genuina.

### 1.3 Descarga del dataset

In [ ]:
!mkdir -p ~/datos
!wget -q -O ~/datos/quijote.txt https://www.gutenberg.org/ebooks/2000.txt.utf-8
!ls -lh ~/datos/quijote.txt

-rw-r--r-- 1 root root 2.2M Sep  1 15:37 /root/datos/quijote.txt


### 1.4 Limpieza de la cabecera y el pie legal

Project Gutenberg anade un bloque de texto legal al inicio y al final de todos sus archivos.
Primero localizamos las marcas reales:

In [ ]:
!grep -n "PROJECT GUTENBERG EBOOK" ~/datos/quijote.txt

27:*** START OF THE PROJECT GUTENBERG EBOOK DON QUIJOTE ***
37709:*** END OF THE PROJECT GUTENBERG EBOOK DON QUIJOTE ***


> **Nota sobre un error cometido:** la primera version de este comando usaba el patron
> `'\*\*\* START OF THE PROJECT GUTENBERG'`. Los asteriscos escapados no coincidieron, `sed` no copio
> ninguna linea y genero un archivo **vacio de 0 bytes**, que provoco el error `The mean is: NaN`
> (division 0/0). La correccion fue quitar los asteriscos del patron.

In [ ]:
!sed -n '/START OF THE PROJECT GUTENBERG/,/END OF THE PROJECT GUTENBERG/p' \
    ~/datos/quijote.txt > ~/datos/quijote_limpio.txt

# Verificacion OBLIGATORIA: el archivo NO debe estar vacio (~37,683 lineas)
!ls -la ~/datos/quijote_limpio.txt
!wc -l ~/datos/quijote_limpio.txt

-rw-r--r-- 1 root root 2206115 Sep 25 16:47 /root/datos/quijote_limpio.txt
37683 /root/datos/quijote_limpio.txt


### 1.5 Carga en HDFS

In [ ]:
!hdfs dfs -mkdir -p /tarea/wordmean/input
!hdfs dfs -put -f ~/datos/quijote_limpio.txt /tarea/wordmean/input/
!hdfs dfs -ls /tarea/wordmean/input

Found 1 items
-rw-r--r--   2 root hadoop    2206115 2026-09-25 16:47 /tarea/wordmean/input/quijote_limpio.txt


### 1.6 Ejecucion del job

**`-D mapreduce.job.reduces=1` es imprescindible aqui.** `WordMean` lee unicamente el archivo
`part-r-00000` para calcular la media. Con varios reducers (el cluster lanzaba 7 por defecto), las
claves `count` y `length` se reparten entre archivos distintos segun su hash, el programa encuentra
solo una y asume que la otra vale 0 &rarr; el resultado sale como **`Infinity`**.

Es una limitacion conocida del ejemplo, escrito asumiendo un unico reducer.

In [ ]:
!hdfs dfs -rm -r -f -skipTrash /tarea/wordmean/output

!hadoop jar /usr/lib/hadoop-mapreduce/hadoop-mapreduce-examples.jar wordmean \
    -D mapreduce.job.reduces=1 \
    /tarea/wordmean/input \
    /tarea/wordmean/output

2026-09-25 16:47:21,935 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at cluster-bigdata-v2-m.us-east4-c.c.eco-tenure-506905-q7.internal./10.150.0.25:8032
2026-09-25 16:47:22,153 INFO client.AHSProxy: Connecting to Application History server at cluster-bigdata-v2-m.us-east4-c.c.eco-tenure-506905-q7.internal./10.150.0.25:10200
2026-09-25 16:47:22,426 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1790353038859_0002
2026-09-25 16:47:22,909 INFO input.FileInputFormat: Total input files to process : 1
2026-09-25 16:47:23,002 INFO mapreduce.JobSubmitter: number of splits:1
2026-09-25 16:47:23,228 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1790353038859_0002
2026-09-25 16:47:23,228 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-09-25 16:47:23,479 INFO conf.Configuration: resource-types.xml not found
2026-09-25 16:47:23,480 INFO resource.ResourceUtils: Unable to fin

### 1.7 Resultados almacenados en HDFS

In [ ]:
!hdfs dfs -ls /tarea/wordmean/output
!echo "--- contenido de part-r-00000 ---"
!hdfs dfs -cat /tarea/wordmean/output/part-r-00000

Found 2 items
-rw-r--r--   2 root hadoop          0 2026-09-25 16:47 /tarea/wordmean/output/_SUCCESS
-rw-r--r--   2 root hadoop         28 2026-09-25 16:47 /tarea/wordmean/output/part-r-00000
--- contenido de part-r-00000 ---
count	386634
length	1718442


### 1.8 Verificacion del calculo

Los valores crudos permiten reproducir la media a mano.

In [ ]:
import subprocess

salida = subprocess.check_output(
    "hdfs dfs -cat /tarea/wordmean/output/part-r-00000", shell=True).decode()

datos = {}
for linea in salida.strip().split("\n"):
    clave, valor = linea.split()
    datos[clave] = int(valor)

print("Palabras totales (count) :", f"{datos['count']:,}")
print("Caracteres totales (length):", f"{datos['length']:,}")
print("Media = length / count     :", datos["length"] / datos["count"])

Palabras totales (count) : 386,634
Caracteres totales (length): 1,718,442
Media = length / count     : 4.444622045655581


### 1.9 Analisis

| Metrica | Valor |
|---|---|
| Lineas procesadas (`Map input records`) | 37,683 |
| Pares emitidos por Map (`Map output records`) | 773,268 |
| **Palabras reales** (`count` en HDFS) | **386,634** |
| Caracteres totales (`length` en HDFS) | 1,718,442 |
| **Longitud media de palabra** | **4.4446 caracteres** |
| Mappers / Reducers | 1 / 1 |

**Cuidado con `Map output records`:** no equivale al numero de palabras. El mapper de `WordMean`
emite **dos pares clave-valor por cada palabra** (uno con la clave `count` y otro con la clave
`length`), asi que 773,268 = 2 x 386,634. El numero real de palabras es el que queda almacenado en
HDFS bajo la clave `count`. Confundir ambas cifras es un error facil de cometer al leer los
contadores de un job.

**Interpretacion:** ~4.44 caracteres por palabra es coherente para el espanol literario, donde el
vocabulario esta dominado por palabras funcionales cortas (articulos, preposiciones, conjunciones)
que arrastran la media hacia abajo pese a la presencia de terminos largos. La densidad tambien
cuadra: 386,634 palabras / 37,683 lineas = 10.3 palabras por linea, tipico de un texto en prosa.

**Observacion sobre el paralelismo:** el archivo pesa ~2.2 MB, muy por debajo del tamano de bloque de
HDFS (128 MB). Por eso se genero un unico *input split* y, en consecuencia, **un solo mapper**
(`number of splits:1`). El resultado es correcto, pero este job no aprovecho el paralelismo del
cluster: el volumen es demasiado pequeno. El contraste con `terasort` (21 mappers) lo deja claro.

---
# 2. Programa 2 &mdash; `secondarysort`

## 2.1 Objetivo

Demostrar el patron de **ordenamiento secundario**. MapReduce garantiza que las *claves* lleguen
ordenadas al reducer, pero **no** ordena los *valores* dentro de cada grupo. El ejemplo lo resuelve
combinando tres piezas:

- una **clave compuesta** (`IntPair`) que contiene los dos enteros;
- un **particionador personalizado** (`FirstPartitioner`) que manda al mismo reducer todos los registros con el mismo primer valor;
- un **comparador de agrupamiento** (`FirstGroupingComparator`) que hace que el reducer trate como un solo grupo los registros que comparten la primera clave, recibiendo los segundos valores ya ordenados.

## 2.2 Por que este dataset

El programa exige **dos enteros por linea**. La opcion facil habria sido escribir a mano un archivo
de 8 lineas, pero eso es un dato sintetico e irrelevante como evidencia de Big Data.

Se busco un dataset **real** con ese patron natural de dos columnas numericas: **MovieLens 100K**
(GroupLens Research, University of Minnesota).

1. Dataset **publico, real y ampliamente citado** en la literatura de sistemas de recomendacion.
2. **100,000 registros**: cuatro ordenes de magnitud mas que un archivo escrito a mano.
3. Su archivo `u.data` tiene el formato `usuario &#8677; pelicula &#8677; calificacion &#8677; timestamp`; quedandose con las dos primeras columnas se obtiene el par requerido **sin alterar ni inventar datos**.
4. El resultado tiene **interpretacion real**: para cada usuario, el catalogo ordenado de peliculas que califico.

### 2.3 Descarga y preparacion del dataset

In [ ]:
!mkdir -p ~/datos
!cd ~/datos && wget -q -O ml-100k.zip https://files.grouplens.org/datasets/movielens/ml-100k.zip
!cd ~/datos && unzip -o -q ml-100k.zip
!ls ~/datos/ml-100k/ | head

README
allbut.pl
mku.sh
u.data
u.genre
u.info
u.item
u.occupation
u.user
u1.base


Formato original de `u.data` (4 columnas separadas por tabulacion):

In [ ]:
!head -5 ~/datos/ml-100k/u.data

196	242	3	881250949
186	302	3	891717742
22	377	1	878887116
244	51	2	880606923
166	346	1	886397596


Nos quedamos con las dos primeras columnas: `(id_usuario, id_pelicula)`.

In [ ]:
!awk '{print $1, $2}' ~/datos/ml-100k/u.data > ~/datos/secondarysort_input.txt

!wc -l ~/datos/secondarysort_input.txt
!head -5 ~/datos/secondarysort_input.txt

100000 /root/datos/secondarysort_input.txt
196 242
186 302
22 377
244 51
166 346


### 2.4 Carga en HDFS

In [ ]:
!hdfs dfs -mkdir -p /tarea/secondarysort/input
!hdfs dfs -put -f ~/datos/secondarysort_input.txt /tarea/secondarysort/input/
!hdfs dfs -ls /tarea/secondarysort/input

Found 1 items
-rw-r--r--   2 root hadoop     779173 2026-09-25 16:48 /tarea/secondarysort/input/secondarysort_input.txt


### 2.5 Ejecucion del job

Aqui `-D mapreduce.job.reduces=1` se usa por un motivo **distinto** al de `wordmean`: con varios
reducers el resultado seria igualmente correcto (cada usuario cae entero en un unico reducer, asi
que ningun grupo se parte), pero la salida se repartiria en 7 archivos `part-r-*`, complicando
explorarla y documentarla. Es una decision de conveniencia, no una correccion de un error.

In [ ]:
!hdfs dfs -rm -r -f -skipTrash /tarea/secondarysort/output

!hadoop jar /usr/lib/hadoop-mapreduce/hadoop-mapreduce-examples.jar secondarysort \
    -D mapreduce.job.reduces=1 \
    /tarea/secondarysort/input \
    /tarea/secondarysort/output

Deleted /tarea/secondarysort/output
2026-09-25 16:48:24,961 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at cluster-bigdata-v2-m.us-east4-c.c.eco-tenure-506905-q7.internal./10.150.0.25:8032
2026-09-25 16:48:25,181 INFO client.AHSProxy: Connecting to Application History server at cluster-bigdata-v2-m.us-east4-c.c.eco-tenure-506905-q7.internal./10.150.0.25:10200
2026-09-25 16:48:25,446 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1790353038859_0003
2026-09-25 16:48:25,846 INFO input.FileInputFormat: Total input files to process : 1
2026-09-25 16:48:25,936 INFO mapreduce.JobSubmitter: number of splits:1
2026-09-25 16:48:26,178 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1790353038859_0003
2026-09-25 16:48:26,178 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-09-25 16:48:26,416 INFO conf.Configuration: resource-types.xml not found
2026-09-25 16:48:26,416 INFO r

### 2.6 Exploracion de resultados

La salida son bloques separados por una linea de guiones, uno por usuario, con los IDs de pelicula
ordenados ascendentemente dentro de cada grupo.

In [ ]:
!hdfs dfs -cat /tarea/secondarysort/output/part-r-00000 2>/dev/null | head -30

------------------------------------------------
1	1
1	2
1	3
1	4
1	5
1	6
1	7
1	8
1	9
1	10
1	11
1	12
1	13
1	14
1	15
1	16
1	17
1	18
1	19
1	20
1	21
1	22
1	23
1	24
1	25
1	26
1	27
1	28
1	29


Conteo de grupos generados (debe dar **943**, el total de usuarios de MovieLens 100K):

In [ ]:
!hdfs dfs -cat /tarea/secondarysort/output/part-r-00000 2>/dev/null | grep -c "^-----"

943


### 2.7 Evidencia del ordenamiento secundario

La prueba mas clara es comparar un mismo usuario antes y despues del procesamiento.

In [ ]:
print("=== ENTRADA (desordenada) — usuario 196 ===")
!grep "^196 " ~/datos/secondarysort_input.txt | head -10

=== ENTRADA (desordenada) — usuario 196 ===
196 242
196 393
196 381
196 251
196 655
196 67
196 306
196 238
196 663
196 111


In [ ]:
print("=== SALIDA (ordenada) — usuario 196 ===")
!hdfs dfs -cat /tarea/secondarysort/output/part-r-00000 2>/dev/null | grep -P "^196\t" | head -10

=== SALIDA (ordenada) — usuario 196 ===
196	8
196	13
196	25
196	66
196	67
196	70
196	94
196	108
196	110
196	111


### 2.8 Analisis

| Metrica | Valor |
|---|---|
| Registros de entrada | 100,000 |
| **Grupos formados** (`Reduce input groups`) | **943** |
| Registros de salida | 100,943 |
| Tamano de entrada en HDFS | 779,173 bytes |

Los **943 grupos** coinciden exactamente con el numero de usuarios del dataset, confirmando que el
agrupamiento por clave primaria funciono. La diferencia entre los 100,000 registros de entrada y los
100,943 de salida corresponde justo a las 943 lineas separadoras que el reducer escribe al inicio de
cada grupo.

Los datos del usuario 196 son los mismos antes y despues, pero MapReduce los reorganizo: agrupados
por usuario (clave primaria) y ordenados por ID de pelicula (clave secundaria). Ese reordenamiento
**no lo hace el codigo del usuario**, sino el framework, gracias al particionador y al comparador
personalizados.

---
# 3. Programa 3 &mdash; `terasort`

## 3.1 Objetivo

`terasort` es el **benchmark estandar de la industria** para medir el rendimiento del ordenamiento
distribuido en clusteres Hadoop (es la prueba usada en las competiciones Sort Benchmark). Se ejecuta
en tres fases encadenadas:

| Fase | Funcion | Entrada | Salida |
|---|---|---|---|
| `teragen` | Genera registros aleatorios de 100 bytes | N&ordm; de registros | `/tarea/terasort/input` |
| `terasort` | Ordena todos los registros por su clave | `input` | `/tarea/terasort/output` |
| `teravalidate` | Verifica que el orden sea correcto | `output` | `/tarea/terasort/validate` |

## 3.2 Por que este dataset

A diferencia de los dos anteriores, aqui **no se eligio el dataset: viene impuesto por el programa**.
`teragen` es el generador oficial disenado especificamente para alimentar a `terasort`. No existe la
alternativa de "traer un archivo real" porque `terasort` espera un formato binario propio: registros
de exactamente 100 bytes, con los primeros 10 bytes como clave aleatoria y el resto como relleno
estructurado.

Se eligieron **1,000,000 de registros (~100 MB)**: suficiente para que el cluster reparta el trabajo
en 21 mappers y 7 reducers, demostrando paralelismo real, y manejable en tiempo de ejecucion. De los
tres programas, este es el que mejor representa un escenario de Big Data.

### 3.3 Fase 1 &mdash; `teragen`

Job **exclusivamente de Map**, sin fase reduce: cada mapper genera su porcion de datos y la escribe
directamente en HDFS. Por eso los archivos llevan el prefijo `part-m-` (map) y no `part-r-` (reduce).

In [ ]:
!hdfs dfs -rm -r -f -skipTrash /tarea/terasort
!hdfs dfs -mkdir -p /tarea/terasort

!time hadoop jar /usr/lib/hadoop-mapreduce/hadoop-mapreduce-examples.jar teragen \
    1000000 /tarea/terasort/input

Deleted /tarea/terasort
2026-09-25 16:49:19,440 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at cluster-bigdata-v2-m.us-east4-c.c.eco-tenure-506905-q7.internal./10.150.0.25:8032
2026-09-25 16:49:19,725 INFO client.AHSProxy: Connecting to Application History server at cluster-bigdata-v2-m.us-east4-c.c.eco-tenure-506905-q7.internal./10.150.0.25:10200
2026-09-25 16:49:20,077 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1790353038859_0004
2026-09-25 16:49:20,449 INFO terasort.TeraGen: Generating 1000000 using 21
2026-09-25 16:49:20,549 INFO mapreduce.JobSubmitter: number of splits:21
2026-09-25 16:49:20,837 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1790353038859_0004
2026-09-25 16:49:20,837 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-09-25 16:49:21,174 INFO conf.Configuration: resource-types.xml not found
2026-09-25 16:49:21,174 INFO resource.ResourceUtils

In [ ]:
!hdfs dfs -ls /tarea/terasort/input | head -8
!echo "--- tamano total ---"
!hdfs dfs -du -s -h /tarea/terasort/input

Found 22 items
-rw-r--r--   2 root hadoop          0 2026-09-25 16:50 /tarea/terasort/input/_SUCCESS
-rw-r--r--   2 root hadoop    4762000 2026-09-25 16:49 /tarea/terasort/input/part-m-00000
-rw-r--r--   2 root hadoop    4761900 2026-09-25 16:49 /tarea/terasort/input/part-m-00001
-rw-r--r--   2 root hadoop    4761900 2026-09-25 16:49 /tarea/terasort/input/part-m-00002
-rw-r--r--   2 root hadoop    4761900 2026-09-25 16:49 /tarea/terasort/input/part-m-00003
-rw-r--r--   2 root hadoop    4761900 2026-09-25 16:49 /tarea/terasort/input/part-m-00004
-rw-r--r--   2 root hadoop    4761900 2026-09-25 16:49 /tarea/terasort/input/part-m-00005
--- tamano total ---
95.4 M  190.7 M  /tarea/terasort/input


> Los "100 MB" y los 95.4 M que reporta `-du -h` no son contradictorios: 100,000,000 bytes en
> base 10 equivalen a 95.4 MiB en base 2.

### 3.4 Fase 2 &mdash; `terasort`

**Aqui NO se fuerza un solo reducer**, a diferencia de los programas 1 y 2. Antes del job propiamente
dicho aparece una fase preparatoria reveladora:

```
Sampling 10 splits of 21
Making 7 from 100000 sampled records
Computing parititions took 727ms
```

Ese es el **`TotalOrderPartitioner`**: muestrea 100,000 claves para determinar los rangos de reparto,
de modo que el reducer 0 recibe las claves mas bajas, el 1 las siguientes, etc. El resultado es que
cada `part-r-*` esta ordenado internamente **y ademas** todo `part-r-00000` &lt; `part-r-00001` &lt; ...
Es decir, se obtiene un **orden global** aunque la salida este repartida en varios archivos. El
paralelismo es la idea central del benchmark.

In [ ]:
!time hadoop jar /usr/lib/hadoop-mapreduce/hadoop-mapreduce-examples.jar terasort \
    /tarea/terasort/input /tarea/terasort/output

2026-09-25 16:50:36,849 INFO terasort.TeraSort: starting
2026-09-25 16:50:38,548 INFO input.FileInputFormat: Total input files to process : 21
Spent 300ms computing base-splits.
Spent 4ms computing TeraScheduler splits.
Computing input splits took 304ms
Sampling 10 splits of 21
Making 7 from 100000 sampled records
Computing parititions took 591ms
Spent 898ms computing partitions.
2026-09-25 16:50:39,444 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at cluster-bigdata-v2-m.us-east4-c.c.eco-tenure-506905-q7.internal./10.150.0.25:8032
2026-09-25 16:50:39,635 INFO client.AHSProxy: Connecting to Application History server at cluster-bigdata-v2-m.us-east4-c.c.eco-tenure-506905-q7.internal./10.150.0.25:10200
2026-09-25 16:50:39,775 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1790353038859_0005
2026-09-25 16:50:39,964 INFO mapreduce.JobSubmitter: number of splits:21
2026-09-25 16:50:40,229 INF

### 3.5 Fase 3 &mdash; `teravalidate`

In [ ]:
!time hadoop jar /usr/lib/hadoop-mapreduce/hadoop-mapreduce-examples.jar teravalidate \
    /tarea/terasort/output /tarea/terasort/validate

2026-09-25 16:52:08,529 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at cluster-bigdata-v2-m.us-east4-c.c.eco-tenure-506905-q7.internal./10.150.0.25:8032
2026-09-25 16:52:08,741 INFO client.AHSProxy: Connecting to Application History server at cluster-bigdata-v2-m.us-east4-c.c.eco-tenure-506905-q7.internal./10.150.0.25:10200
2026-09-25 16:52:09,012 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1790353038859_0006
2026-09-25 16:52:09,387 INFO input.FileInputFormat: Total input files to process : 7
Spent 71ms computing base-splits.
Spent 3ms computing TeraScheduler splits.
2026-09-25 16:52:09,462 INFO mapreduce.JobSubmitter: number of splits:7
2026-09-25 16:52:09,725 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1790353038859_0006
2026-09-25 16:52:09,725 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-09-25 16:52:09,960 INFO conf.Configuration: resource-types.xml 

**Criterio de exito:** una unica linea con el checksum y ninguna linea de error. Si algun registro
estuviera fuera de secuencia, `teravalidate` habria escrito una linea adicional por cada violacion
del orden (*misorder*). La validacion cubre tanto el orden interno de cada archivo como los limites
entre archivos consecutivos.

In [ ]:
!hdfs dfs -cat /tarea/terasort/validate/part-r-00000

checksum	7a27e2d0d55de


### 3.6 Estructura de la salida

Los 7 archivos `part-r-*` (uno por reducer) mas `_partition.lst`, que es la tabla de rangos calculada
en la fase de muestreo.

> La columna de replicacion muestra `1` en lugar de `2`. No es un fallo: `terasort` escribe su salida
> con factor de replicacion 1 por defecto, para no penalizar la medicion de rendimiento.

In [ ]:
!hdfs dfs -ls /tarea/terasort/output
!hdfs dfs -du -s -h /tarea/terasort/output

Found 9 items
-rw-r--r--   1 root hadoop          0 2026-09-25 16:52 /tarea/terasort/output/_SUCCESS
-rw-r--r--  10 root hadoop         66 2026-09-25 16:50 /tarea/terasort/output/_partition.lst
-rw-r--r--   1 root hadoop   14363800 2026-09-25 16:51 /tarea/terasort/output/part-r-00000
-rw-r--r--   1 root hadoop   14213400 2026-09-25 16:51 /tarea/terasort/output/part-r-00001
-rw-r--r--   1 root hadoop   14182000 2026-09-25 16:51 /tarea/terasort/output/part-r-00002
-rw-r--r--   1 root hadoop   14120400 2026-09-25 16:51 /tarea/terasort/output/part-r-00003
-rw-r--r--   1 root hadoop   14520500 2026-09-25 16:51 /tarea/terasort/output/part-r-00004
-rw-r--r--   1 root hadoop   14161000 2026-09-25 16:51 /tarea/terasort/output/part-r-00005
-rw-r--r--   1 root hadoop   14438900 2026-09-25 16:52 /tarea/terasort/output/part-r-00006
95.4 M  95.4 M  /tarea/terasort/output


### 3.7 Analisis

#### `teragen`
| Metrica | Valor |
|---|---|
| Registros generados | 1,000,000 |
| Bytes escritos | 100,000,000 (95.4 MiB) |
| Archivos de salida | 21 x `part-m-*` |
| Mappers / Reducers | 21 / **0** |
| Tiempo real | 1m 10.888s |

#### `terasort`
| Metrica | Valor |
|---|---|
| Registros de entrada / salida | 1,000,000 / 1,000,000 |
| Mappers / Reducers | 21 / **7** |
| Tiempo real | 1m 29.969s |

#### `teravalidate`
| Metrica | Valor |
|---|---|
| Registros leidos | 1,000,000 |
| Registros emitidos por Map | 21 (uno por archivo validado) |
| Mappers / Reducers | 7 / 1 |
| Tiempo real | 0m 45.019s |

Que `Map input records` y `Reduce output records` sean ambos exactamente 1,000,000 en la fase de
ordenamiento confirma que no se perdio ni se duplico ningun registro durante el shuffle.

En `teravalidate` los 21 `Map output records` corresponden a un resumen por cada archivo de entrada
validado; el unico `Reduce output record` es la linea del checksum.

---
# 4. Conclusiones

### Sobre el uso de HDFS

En los tres programas, tanto la entrada como la salida residieron en HDFS. Los datasets externos
(Quijote, MovieLens) se descargaron primero al disco local del nodo maestro y desde alli se cargaron
con `hdfs dfs -put`; el de `terasort` se genero directamente en HDFS.

Un punto importante: HDFS vive en los discos de las VMs del cluster, asi que **al eliminar el cluster
los datos de `/tarea` se pierden**. Para conservarlos hay que copiarlos antes al bucket de Cloud
Storage, que es persistente e independiente del ciclo de vida del cluster.

In [ ]:
# Opcional: preservar los resultados en Cloud Storage antes de borrar el cluster
BUCKET = "bucketjamp2006"

!hadoop distcp /tarea/wordmean/output       gs://{BUCKET}/resultados/wordmean
!hadoop distcp /tarea/secondarysort/output  gs://{BUCKET}/resultados/secondarysort
!hadoop distcp /tarea/terasort/validate     gs://{BUCKET}/resultados/terasort

### Sobre el paralelismo y el tamano de los datos

El contraste entre los tres programas es ilustrativo:

| Programa | Tamano de entrada | Mappers |
|---|---|---|
| `secondarysort` | 780 KB | 1 |
| `wordmean` | 2 MB | 1 |
| `terasort` | 100 MB | **21** |

El tamano de bloque por defecto de HDFS (128 MB) determina cuantos *input splits* se generan y, por
tanto, cuanto paralelismo es posible. Un cluster de 3 nodos no acelera un archivo de 2 MB; el
beneficio aparece cuando el volumen justifica la distribucion.

### Sobre el numero de reducers

El hallazgo mas valioso de la practica fue descubrir que la configuracion de reducers no es un
detalle irrelevante, y que su efecto correcto **depende de como este escrito cada programa**:

| Programa | Con varios reducers | Motivo |
|---|---|---|
| `wordmean` | **Rompe el resultado** (`Infinity`) | El codigo lee solo `part-r-00000`; las dos claves quedan separadas |
| `secondarysort` | Solo fragmenta la salida | Cada grupo cae entero en un reducer; correcto pero incomodo de revisar |
| `terasort` | **Es la idea central** | El `TotalOrderPartitioner` mantiene el orden global entre archivos |

### Sobre la depuracion en entornos distribuidos

Los dos fallos de `wordmean` (`NaN` e `Infinity`) ensenaron que en MapReduce un job puede terminar con
`completed successfully` y aun asi producir un resultado invalido. Los **contadores del job**
(`Map input records`, `Bytes Read`, `Launched reduce tasks`) resultaron ser la herramienta de
diagnostico decisiva: fueron ellos, y no los mensajes de error, los que permitieron distinguir entre
"el archivo de entrada estaba vacio" y "la salida se repartio entre varios archivos".

---

## Fuentes de los datasets

- **Don Quijote de la Mancha** &mdash; Project Gutenberg, eBook #2000, texto plano UTF-8. Dominio publico.
- **MovieLens 100K** &mdash; GroupLens Research, University of Minnesota. 100,000 calificaciones de 943 usuarios sobre 1,682 peliculas.
- **TeraSort** &mdash; Datos sinteticos generados por `teragen`, incluido en `hadoop-mapreduce-examples.jar`.